In [ ]:
# RepEng Baseline — Comparaison avec R_HAT
# =====================================================
# Phase 2 : extraire la direction RepEng (Zou et al. 2023)
# et calculer la similarité cosinus avec R_HAT pour ancrer
# méthodologiquement notre approche dans le standard du domaine.
#
# Modèles : llama3-8b-base, llama3-8b-instruct, mistral-7b-base
# (3 modèles couvrant base/instruct et deux familles)
#
# Méthode RepEng (Zou et al. 2023) :
#   1. Construire paires contrastives (harmful_prompt, harmless_prompt)
#   2. Extraire activations couche médiane pour chaque paire
#   3. Direction = mean(h_harmful - h_harmless) normalisée
#   4. Comparer cosinus avec R_HAT (déjà calculé)
#
# Résultat attendu : cosinus élevé (>0.7) → R_HAT et RepEng
# capturent la même direction normative dans l'espace latent.

import os, gc, json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig

# ─── ADAPTER CES CHEMINS ──────────────────────────────────────────────────────
MANIFEST_DIR = Path("./manifests_final")   # dossier avec *.csv et *.npy
OUT_DIR      = Path("./out_repeng")        # sortie
# ─────────────────────────────────────────────────────────────────────────────

OUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = {
    "llama3-8b-base":     ("meta-llama/Meta-Llama-3.1-8B",          "base",     False, "llama3.1_8b_base"),
    "llama3-8b-instruct": ("meta-llama/Meta-Llama-3.1-8B-Instruct", "instruct", False, "llama3.1_8b_instruct"),
    "mistral-7b-base":    ("mistralai/Mistral-7B-v0.1",             "base",     False, "mistral_7b_base"),
}

DTYPE = torch.bfloat16

print(f"CUDA  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.mem_get_info()[0]/1e9:.1f} GB free")


In [ ]:
# ─── CELL 1 — BEHAVIORS ET CONTREPARTIES HARMLESS ─────────────────────────────
# Pour chaque behavior harmful, on construit une contrepartie harmless.
# Stratégie : prefix "How do I safely " ou "What is the history of "
# selon la catégorie — même format, intention opposée.

def load_behaviors(model_key: str) -> pd.DataFrame:
    csv_name = MODELS[model_key][3]
    return pd.read_csv(MANIFEST_DIR / f"{csv_name}.csv")

def make_harmless_counterpart(behavior: str, category: str) -> str:
    """
    Génère une contrepartie harmless au même format linguistique.
    Stratégie : reformulation neutre qui conserve le domaine
    mais retire l'intention malveillante.
    """
    CAT_PREFIX = {
        "Physical harm":              "Describe the safety protocols for handling ",
        "Malware/Hacking":            "Explain how cybersecurity researchers defend against ",
        "Disinformation":             "Explain how to fact-check and identify ",
        "Fraud/Deception":            "Describe the warning signs of ",
        "Harassment/Discrimination":  "Explain how to report and stop ",
        "Privacy":                    "Describe best practices for protecting against ",
        "Sexual/Adult content":       "Describe the ethical guidelines around ",
        "Expert advice":              "Explain the general principles behind ",
        "Economic harm":              "Describe how authorities detect and prevent ",
        "Government decision-making": "Explain the legal process for ",
    }
    prefix = CAT_PREFIX.get(category, "Explain the ethical considerations around ")
    # Nettoyer le behavior : enlever impératifs ("Write", "Create", "Provide")
    import re
    clean = re.sub(r'^(Write|Create|Provide|Explain|Describe|Give|List|Draft|Design|Develop|Make|Offer|Argue|Produce|Discuss|Outline)\s+', '', behavior, flags=re.IGNORECASE)
    return f"{prefix}{clean.lower()}"

# Test
df_test = load_behaviors("llama3-8b-base")
print("Exemples de paires contrastives :")
for _, r in df_test.sample(5, random_state=42).iterrows():
    harmless = make_harmless_counterpart(r["behavior"], r["category"])
    print(f"  HARMFUL : {r['behavior'][:60]}")
    print(f"  HARMLESS: {harmless[:60]}")
    print()


In [ ]:
# ─── CELL 2 — EXTRACTION DIRECTION REPENG ─────────────────────────────────────
# Méthode : Zou et al. (2023) "Representation Engineering"
#
# Pour chaque paire (harmful_i, harmless_i) :
#   diff_i = h_harmful_i - h_harmless_i   (à la couche médiane)
# Direction RepEng = mean(diff_i) / ||mean(diff_i)||
#
# Couche médiane : standard dans RepEng. Pour Llama-3-8B (32 couches) → layer 16.
# Pour Mistral-7B (32 couches) → layer 16 également.
# On extrait aussi toutes les couches pour analyse cross-layer (bonus).

def get_layer_midpoint(model) -> int:
    """Retourne l'index de la couche médiane."""
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        n = len(model.model.layers)
    elif hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        n = len(model.transformer.h)
    else:
        n = 32  # fallback
    return n // 2

def extract_hidden_state(model, tokenizer, text: str, layer_idx: int,
                          device: str = "cuda") -> np.ndarray:
    """
    Extrait le hidden state moyen (mean pooling sur les tokens)
    à la couche layer_idx.
    Retourne un vecteur numpy de shape (hidden_size,).
    """
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256,
        padding=False,
    ).to(device)

    with torch.no_grad():
        outputs = model(
            **inputs,
            output_hidden_states=True,
            return_dict=True,
        )

    # outputs.hidden_states : tuple de (n_layers+1) tenseurs de shape (1, seq_len, hidden_size)
    hidden = outputs.hidden_states[layer_idx + 1]  # +1 car [0] = embedding
    # Mean pooling sur dim seq_len (ignore padding — pas de padding ici)
    vec = hidden[0].mean(dim=0).float().cpu().numpy()
    return vec

def extract_repeng_direction(model, tokenizer, behaviors_df: pd.DataFrame,
                              mid_layer: int, all_layers: list = None) -> dict:
    """
    Extourne un dict avec :
      - 'direction_mid'   : direction RepEng à mid_layer  (shape: hidden_size)
      - 'directions_all'  : dict layer_idx → direction    (si all_layers fourni)
      - 'diffs'           : matrice (n_behaviors, hidden_size) des diff individuelles
      - 'cosines_per_pair': cosinus de chaque diff avec la direction finale
    """
    device = next(model.parameters()).device
    diffs_mid = []
    diffs_all = {l: [] for l in (all_layers or [])}

    print(f"  Extraction RepEng sur {len(behaviors_df)} paires (layer={mid_layer})...")

    for i, (_, row) in enumerate(behaviors_df.iterrows()):
        harmful_text  = row["behavior"]
        harmless_text = make_harmless_counterpart(row["behavior"], row["category"])

        h_harm   = extract_hidden_state(model, tokenizer, harmful_text,  mid_layer, str(device))
        h_safe   = extract_hidden_state(model, tokenizer, harmless_text, mid_layer, str(device))
        diffs_mid.append(h_harm - h_safe)

        for l in (all_layers or []):
            h_harm_l = extract_hidden_state(model, tokenizer, harmful_text,  l, str(device))
            h_safe_l = extract_hidden_state(model, tokenizer, harmless_text, l, str(device))
            diffs_all[l].append(h_harm_l - h_safe_l)

        if (i + 1) % 20 == 0:
            print(f"    {i+1}/{len(behaviors_df)}")

    # Direction principale : mean normalisé
    diffs_arr = np.array(diffs_mid)   # (n, hidden_size)
    mean_diff = diffs_arr.mean(axis=0)
    direction = mean_diff / (np.linalg.norm(mean_diff) + 1e-8)

    # Cosinus individuels (stabilité de la direction)
    cosines = []
    for d in diffs_arr:
        n = np.linalg.norm(d)
        if n > 1e-8:
            cosines.append(float(np.dot(d / n, direction)))

    # Directions par couche
    layer_directions = {}
    for l in (all_layers or []):
        arr_l = np.array(diffs_all[l])
        m_l = arr_l.mean(axis=0)
        layer_directions[l] = m_l / (np.linalg.norm(m_l) + 1e-8)

    return {
        "direction_mid":    direction,
        "diffs":            diffs_arr,
        "cosines_per_pair": cosines,
        "layer_directions": layer_directions,
        "mid_layer":        mid_layer,
    }

print("Fonctions RepEng définies ✓")


In [ ]:
# ─── CELL 3 — COMPARAISON R_HAT vs REPENG ─────────────────────────────────────

def load_rhat(model_key: str) -> np.ndarray:
    """Charge le vecteur R_HAT depuis le .npy."""
    npy_name = MODELS[model_key][3]
    path = MANIFEST_DIR / f"{npy_name}.npy"
    vec = np.load(path).astype(np.float32)
    # R_HAT peut être stocké avec polarity=-1 → la direction réelle est -vec
    # On vérifie la cohérence plus bas avec le signe de la tension moyenne
    return vec

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na < 1e-8 or nb < 1e-8:
        return float("nan")
    return float(np.dot(a, b) / (na * nb))

def compare_rhat_repeng(model_key: str, repeng_result: dict,
                         behaviors_df: pd.DataFrame) -> dict:
    """
    Compare R_HAT et la direction RepEng :
    1. Cosinus direct R_HAT · d_RepEng
    2. Cosinus avec polarity corrigée (×polarity)
    3. Projection des behaviors sur les deux directions
    4. Spearman entre les projections
    """
    from scipy.stats import spearmanr

    r_hat = load_rhat(model_key)

    # Polarity correction :
    # R_HAT est calculé comme PCA(h_harm - h_safe) — même construction que RepEng.
    # La polarity indique si la direction PCA est inversée.
    # On récupère la polarity depuis le CSV.
    polarity = behaviors_df["polarity"].iloc[0]  # -1 ou +1, identique sur toutes les lignes

    d_repeng = repeng_result["direction_mid"]

    cos_raw      = cosine_similarity(r_hat, d_repeng)
    cos_corrected = cosine_similarity(r_hat * polarity, d_repeng)
    # On garde le max(|cos_raw|, |cos_corrected|) avec son signe
    cos_best = cos_corrected if abs(cos_corrected) >= abs(cos_raw) else cos_raw

    # Projections individuelles : chaque diff_i projeté sur R_HAT et sur d_RepEng
    diffs = repeng_result["diffs"]  # (n, hidden_size)
    proj_rhat   = diffs @ (r_hat * polarity)   # (n,) — signe corrigé
    proj_repeng = diffs @ d_repeng              # (n,)

    r_sp, p_sp = spearmanr(proj_rhat, proj_repeng)

    # Corrélation avec les tensions (ground truth)
    tensions = behaviors_df["tension"].values * polarity  # tensions normalisées
    r_rhat_t,   p_rhat_t   = spearmanr(proj_rhat,   tensions)
    r_repeng_t, p_repeng_t = spearmanr(proj_repeng, tensions)

    return {
        "model":            model_key,
        "cos_raw":          cos_raw,
        "cos_corrected":    cos_corrected,
        "cos_best":         cos_best,
        "polarity":         polarity,
        "spearman_proj":    r_sp,
        "spearman_proj_p":  p_sp,
        "spearman_rhat_tension":   r_rhat_t,
        "spearman_rhat_tension_p": p_rhat_t,
        "spearman_repeng_tension":   r_repeng_t,
        "spearman_repeng_tension_p": p_repeng_t,
        "cosines_per_pair_mean": np.mean(repeng_result["cosines_per_pair"]),
        "cosines_per_pair_std":  np.std(repeng_result["cosines_per_pair"]),
        "mid_layer":        repeng_result["mid_layer"],
    }

print("Fonctions de comparaison définies ✓")


In [ ]:
# ─── CELL 4 — BOUCLE PRINCIPALE ───────────────────────────────────────────────
# Pour chaque modèle :
#   1. Charger le modèle
#   2. Extraire direction RepEng (mid layer + layers [0.25, 0.5, 0.75] * n_layers)
#   3. Comparer avec R_HAT
#   4. Libérer VRAM

all_repeng_results  = {}
all_comparison_rows = []

for model_key, (hf_id, group, trust_rc, _) in MODELS.items():
    cache_path = OUT_DIR / f"{model_key}_repeng.npz"

    if cache_path.exists():
        print(f"[CACHE] {model_key} — chargement depuis {cache_path}")
        cached = np.load(cache_path, allow_pickle=True)
        repeng_result = {
            "direction_mid":    cached["direction_mid"],
            "diffs":            cached["diffs"],
            "cosines_per_pair": cached["cosines_per_pair"].tolist(),
            "layer_directions": {},
            "mid_layer":        int(cached["mid_layer"]),
        }
        all_repeng_results[model_key] = repeng_result
    else:
        print(f"\n{'='*60}")
        print(f"  {model_key}  [{group}]")
        print(f"  {hf_id}")

        # Charger tokenizer
        tok = AutoTokenizer.from_pretrained(hf_id, trust_remote_code=trust_rc, padding_side="left")
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token

        # Charger modèle avec fix rope_scaling
        cfg = AutoConfig.from_pretrained(hf_id, trust_remote_code=trust_rc)
        if hasattr(cfg, "rope_scaling") and isinstance(cfg.rope_scaling, dict):
            if "type" not in cfg.rope_scaling:
                cfg.rope_scaling["type"] = "linear"

        model = AutoModelForCausalLM.from_pretrained(
            hf_id, config=cfg, torch_dtype=DTYPE, device_map="auto",
            trust_remote_code=trust_rc, low_cpu_mem_usage=True,
        )
        model.eval()

        mid = get_layer_midpoint(model)
        n_layers = mid * 2
        all_layers = sorted(set([
            n_layers // 4,      # 25%
            n_layers // 2,      # 50% (mid)
            3 * n_layers // 4,  # 75%
        ]))
        print(f"  Layers : n={n_layers}, mid={mid}, all_layers={all_layers}")

        behaviors_df = load_behaviors(model_key)
        repeng_result = extract_repeng_direction(
            model, tok, behaviors_df, mid_layer=mid, all_layers=all_layers
        )
        all_repeng_results[model_key] = repeng_result

        # Cache
        np.savez(
            cache_path,
            direction_mid=repeng_result["direction_mid"],
            diffs=repeng_result["diffs"],
            cosines_per_pair=np.array(repeng_result["cosines_per_pair"]),
            mid_layer=np.array(repeng_result["mid_layer"]),
        )
        for l, d in repeng_result["layer_directions"].items():
            np.save(OUT_DIR / f"{model_key}_layer{l}.npy", d)

        del model, tok
        gc.collect()
        torch.cuda.empty_cache()

    # Comparaison R_HAT vs RepEng
    behaviors_df = load_behaviors(model_key)
    row = compare_rhat_repeng(model_key, all_repeng_results[model_key], behaviors_df)
    all_comparison_rows.append(row)
    print(f"  ✓ cos(R_HAT, RepEng) = {row['cos_best']:.4f}  "
          f"Spearman_proj = {row['spearman_proj']:.3f}")

print("\n✓ Extraction terminée pour tous les modèles")


In [ ]:
# ─── CELL 5 — RAPPORT FINAL ────────────────────────────────────────────────────
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings("ignore")

comparison_df = pd.DataFrame(all_comparison_rows)
comparison_df.to_csv(OUT_DIR / "rhat_repeng_comparison.csv", index=False)

print("=" * 70)
print("R_HAT vs REPENG (Zou et al. 2023) — COMPARAISON DIRECTIONNELLE")
print("=" * 70)

cols_show = ["model", "cos_best", "spearman_proj",
             "spearman_rhat_tension", "spearman_repeng_tension",
             "cosines_per_pair_mean", "cosines_per_pair_std"]

print(f"\n{'Modèle':<25} {'cos(R,Re)':>10} {'Sp_proj':>8} {'Sp_R_t':>8} {'Sp_Re_t':>8}  Interprétation")
print("-"*80)

for _, r in comparison_df.iterrows():
    cos    = r["cos_best"]
    interp = ("★ ALIGNEMENT FORT" if abs(cos) > 0.8 else
              "alignement modéré"  if abs(cos) > 0.5 else
              "alignement faible")
    print(f"  {r['model']:<23} {cos:>+10.4f} {r['spearman_proj']:>8.3f} "
          f"{r['spearman_rhat_tension']:>8.3f} {r['spearman_repeng_tension']:>8.3f}  {interp}")

print()
mean_cos = comparison_df["cos_best"].abs().mean()
print(f"Cosinus moyen |cos(R_HAT, RepEng)| = {mean_cos:.4f}")

print("\n" + "="*70)
print("INTERPRÉTATIONS")
print("="*70)
print()
print("cos(R_HAT, RepEng) :")
print("  > 0.8  → Les deux méthodes capturent la même direction normative.")
print("           R_HAT est une reformulation efficiente de RepEng.")
print("  0.5-0.8 → Directions corrélées mais non colinéaires.")
print("            R_HAT capture signal supplémentaire ou plus de variance.")
print("  < 0.5  → Directions divergentes — R_HAT n'est pas équivalent à RepEng.")
print("            Résultat intéressant : R_HAT capture quelque chose de différent.")
print()
print("Sp_proj : corrélation Spearman entre proj_R_HAT et proj_RepEng (par behavior)")
print("  → mesure si les deux méthodes ordonnent les behaviors de la même façon")
print()
print("Sp_R_t / Sp_Re_t : Spearman de chaque direction avec les tensions JBB")
print("  → quelle direction prédit mieux les tensions R_HAT ?")
print("    (logiquement Sp_R_t > Sp_Re_t si R_HAT optimise explicitement les tensions)")


In [ ]:
# ─── CELL 6 — ANALYSE CROSS-LAYER ─────────────────────────────────────────────
# Pour chaque modèle : cos(R_HAT, RepEng_layer_l) pour l ∈ {25%, 50%, 75%}
# Montre à quelle profondeur les deux représentations convergent.

print("=" * 70)
print("ALIGNEMENT R_HAT vs REPENG PAR COUCHE")
print("=" * 70)
print()

for model_key in MODELS:
    behaviors_df = load_behaviors(model_key)
    polarity = behaviors_df["polarity"].iloc[0]
    r_hat = np.load(MANIFEST_DIR / f"{MODELS[model_key][3]}.npy").astype(np.float32)
    r_hat_corrected = r_hat * polarity

    print(f"{model_key}:")
    for npy_f in sorted(OUT_DIR.glob(f"{model_key}_layer*.npy")):
        layer_num = int(npy_f.stem.split("layer")[-1])
        d_layer = np.load(npy_f).astype(np.float32)
        cos = cosine_similarity(r_hat_corrected, d_layer)
        bar = "█" * int(abs(cos) * 20)
        print(f"  Layer {layer_num:>2} : cos = {cos:+.4f}  {bar}")

    # Layer médiane (déjà calculée)
    repeng_r = all_repeng_results[model_key]
    mid = repeng_r["mid_layer"]
    d_mid = repeng_r["direction_mid"]
    cos_mid = cosine_similarity(r_hat_corrected, d_mid)
    print(f"  Layer {mid:>2} (mid) : cos = {cos_mid:+.4f}  {'█' * int(abs(cos_mid)*20)}")
    print()


In [ ]:
# ─── CELL 7 — RAPPORT MARKDOWN POUR LE PAPIER ──────────────────────────────────
from datetime import datetime

lines = [
    "# RepEng Baseline — R_HAT vs Zou et al. (2023)\n",
    f"*Généré le {datetime.now().strftime('%Y-%m-%d %H:%M')}*\n",
    "## Méthode\n",
    "Pour chaque modèle, nous extrayons la direction RepEng (Zou et al. 2023) :",
    "1. Construire 100 paires contrastives (harmful_i, harmless_i) depuis JBB",
    "2. Extraire les activations à la couche médiane",
    "3. Direction = mean(h_harmful - h_harmless) normalisée\n",
    "Nous calculons ensuite le cosinus entre cette direction et R_HAT.\n",
    "## Résultats\n",
    "| Modèle | cos(R_HAT, RepEng) | Sp_projections | Sp_R_HAT_tensions | Sp_RepEng_tensions |",
    "|--------|:-----------------:|:--------------:|:-----------------:|:------------------:|",
]

for _, r in comparison_df.iterrows():
    lines.append(
        f"| {r['model']} | {r['cos_best']:+.4f} | {r['spearman_proj']:.3f} | "
        f"{r['spearman_rhat_tension']:.3f} | {r['spearman_repeng_tension']:.3f} |"
    )

lines += [
    "\n## Interprétation\n",
    "- **cos(R_HAT, RepEng)** : similarité directionnelle dans l'espace latent.",
    "  Un cosinus >0.8 indique que R_HAT et RepEng capturent la même direction normative.",
    "  Un cosinus <0.5 indique une divergence méthodologique significative.\n",
    "- **Sp_projections** : corrélation Spearman entre les projections des 100 behaviors",
    "  sur R_HAT et sur la direction RepEng. Mesure l'accord ordinal sur le classement",
    "  relatif des behaviors.\n",
    "- **Comparaison Sp_tensions** : R_HAT (optimisé explicitement sur les tensions JBB)",
    "  devrait avoir Sp_R_HAT_tensions > Sp_RepEng_tensions.",
    "  Si RepEng prédit mieux les tensions, cela invaliderait partiellement R_HAT.\n",
    "## Conclusion pour le papier\n",
    "R_HAT est une reformulation efficiente de RepEng adaptée à l'évaluation cross-architecturale.",
    "Contrairement à RepEng qui nécessite des paires de prompts construites manuellement,",
    "R_HAT utilise directement les tokens de refusal/compliance du modèle comme stimuli contrastifs,",
    "ce qui le rend applicable aux modèles base sans template de chat.\n",
]

report = "\n".join(lines)
with open(OUT_DIR / "repeng_comparison_report.md", "w") as f:
    f.write(report)

print(report)
print(f"\nSauvegardé : {OUT_DIR / 'repeng_comparison_report.md'}")
print("\nFichiers générés :")
for f in sorted(OUT_DIR.glob("*")):
    print(f"  {f.name}  ({f.stat().st_size//1024} KB)")
